# Oversampling

### <i> Using SDV Framework

<br>
<br>

`SDV` learns the overall data distribution and generates synthetic records from that learned distribution. You have a wide variety of options (Synthesizer algorithms) you can work with when deciding how you want to create more data:

</i>
<br>

|Synthesizer|Approach|
|---|---|
|GaussianCopulaSynthesizer| statistical/correlation-based approach (REALLY robust)|
|CTGAN| neural network-based|
|TVAE| variational autoencoder approach (good for noise)|
|CopulaGAN| GAN + copula approach (handles huge ranges in values decently well)|


<i>



<i> Not bad. You can see CTGAN struggle with thin and thick-tailed distributions. You can also see it struggles to mimic the original dataset when big ranges are involved.

#### Gaussian Copula - classification example

You can also use other methods like TVAE or CTGAN for classification!

In [12]:
from sdv.metadata import Metadata
from sdv.single_table import GaussianCopulaSynthesizer
import pandas as pd

# load data
df = pd.read_csv("../../lifestyle_sustainability_data.csv")

df.columns

Index(['ParticipantID', 'Age', 'Location', 'DietType', 'LocalFoodFrequency',
       'TransportationMode', 'EnergySource', 'HomeType', 'HomeSize',
       'ClothingFrequency', 'SustainableBrands', 'EnvironmentalAwareness',
       'CommunityInvolvement', 'MonthlyElectricityConsumption',
       'MonthlyWaterConsumption', 'Gender', 'UsingPlasticProducts',
       'DisposalMethods', 'PhysicalActivities', 'Rating'],
      dtype='object')

In [13]:
# Create the metadata again
metadata = Metadata.detect_from_dataframe(
    data=df
)

# Place the metadata into the GaussianCopula model approach
synthesizer = GaussianCopulaSynthesizer(
    metadata
)

# Set and fit the data into our dataframe.
synthesizer.fit(df)

# Make 500 rows using Gaussian Copula
synthetic_df = synthesizer.sample(
    num_rows=500
)

c:\Users\tuomas.valtanen\dataeng_advda2026_lectures\DataEngineering_AdvDA_2026\.venv\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


In [14]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# # Let's make two histplots for each column. One for the Original dataset, the other for CTGAN.
# for column in df.columns:

#     # Create the canvas and create
#     fig, axes = plt.subplots(1, 2, figsize=(12, 5))
#     sns.histplot(df[column], ax=axes[0], color="red")
#     sns.histplot(synthetic_data[column], ax=axes[1], color="purple")

#     axes[0].set_title("Original")
#     axes[1].set_title("Gaussian Copula Synthetic")



###  Using `SDV` evaluate Quality

In [15]:
# Let's use the evaluate_quality function in SDV's library
from sdv.evaluation.single_table import evaluate_quality

quality_report = evaluate_quality(
    real_data=df,
    synthetic_data=synthetic_df,
    metadata=metadata
)

# The higher the percentile, the better the fit. 
quality_report

# Column shape: Compares real data vs synthetic data distributions.
# Column pairs: Compares pairs, ie real x: income y: age vs. synthetic x: income y:age

# good values are probably in the 80-95% range, we shouldn't expect 100%

C:\Users\tuomas.valtanen\AppData\Local\Temp\ipykernel_4764\3912371679.py:2: FutureWarning: The evaluation functions are now accessible via the 'sdv.evaluation' module.
  from sdv.evaluation.single_table import evaluate_quality


Generating report ...

(1/4) Evaluating Column Shapes: |██████████| 20/20 [00:00<00:00, 1691.22it/s]|
Column Shapes Score: 96.12%

(2/4) Evaluating Column Pair Trends: |██████████| 190/190 [00:00<00:00, 425.69it/s]|
Column Pair Trends Score: 75.26%

(3/4) Evaluating Cardinality: N/A
This property does not apply to single-table data.

(4/4) Evaluating Intertable Trends: N/A
This property does not apply to single-table data.

Overall Score (Average): 85.69%



---

In [16]:
# switching to decimal notation for the variance checks
pd.set_option('display.float_format', lambda x: "%.2f" % x)

In [17]:
# We can compare the statistics between the two dataframes by using `.describe()`
df.describe()

,ParticipantID,Age,HomeSize,EnvironmentalAwareness,MonthlyElectricityConsumption,MonthlyWaterConsumption,Rating
count,499.00,499.00,499.00,499.00,499.00,499.00,499.00
mean,250.02,44.05,1518.05,3.06,291.96,3139.52,3.43
std,144.23,14.90,672.21,1.34,120.48,1224.61,1.50
min,1.00,18.00,407.00,1.00,55.00,536.00,1.00
25%,125.50,31.00,951.50,2.00,185.00,2000.00,2.00
50%,250.00,44.00,1450.00,3.00,295.00,3350.00,4.00
75%,374.50,58.00,1995.00,4.00,400.00,4100.00,5.00
max,500.00,96.00,2997.00,5.00,498.00,5250.00,5.00


In [18]:
# pretty similar!
synthetic_df.describe()

,ParticipantID,Age,HomeSize,EnvironmentalAwareness,MonthlyElectricityConsumption,MonthlyWaterConsumption,Rating
count,500.00,500.00,500.00,500.00,500.00,500.00,500.00
mean,8384606.68,43.14,1491.17,3.04,304.89,3111.89,3.47
std,4884787.44,15.07,665.88,1.38,123.20,1213.86,1.47
min,62880.00,18.00,414.00,1.00,61.00,605.00,1.00
25%,4030316.75,31.75,923.50,2.00,200.00,2068.00,2.00
50%,8645443.50,41.00,1431.00,3.00,314.00,3247.00,4.00
75%,12603413.50,53.00,1950.50,4.00,415.00,4106.50,5.00
max,16753185.00,93.00,2997.00,5.00,498.00,5213.00,5.00


In [19]:
synthetic_df

,ParticipantID,Age,Location,DietType,LocalFoodFrequency,TransportationMode,EnergySource,HomeType,HomeSize,ClothingFrequency,SustainableBrands,EnvironmentalAwareness,CommunityInvolvement,MonthlyElectricityConsumption,MonthlyWaterConsumption,Gender,UsingPlasticProducts,DisposalMethods,PhysicalActivities,Rating
0,7737806,54,Suburban,Balanced,Rarely,Public Transit,Renewable,House,2586,Always,False,1,Moderate,429,4419,Female,Rarely,Recycling,NaN,1
1,16599194,37,Suburban,Mostly Animal-Based,Sometimes,Walk,Non-Renewable,Apartment,1833,Always,True,1,Moderate,169,2939,Male,Sometimes,Recycling,NaN,4
2,2811854,39,Suburban,Mostly Animal-Based,Always,Bike,Non-Renewable,Other,1658,Often,False,2,NaN,409,3563,Male,Sometimes,Combination,Moderate,3
3,15703660,49,Rural,Mostly Animal-Based,Often,Bike,Mixed,Apartment,1751,Always,True,2,High,275,4768,Prefer not to say,Rarely,Landfill,Low,5
4,1846724,62,Suburban,Mostly Animal-Based,Always,Car,Mixed,House,1737,Often,False,1,NaN,479,5159,Non-Binary,Never,Composting,NaN,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,2085997,29,Urban,Mostly Plant-Based,Often,Public Transit,Non-Renewable,Apartment,917,Rarely,False,2,High,248,2017,Male,Sometimes,Composting,High,3
496,1197143,32,Urban,Mostly Plant-Based,Sometimes,Public Transit,Renewable,House,513,Rarely,True,5,High,180,3252,Male,Never,Composting,High,5
497,14511357,45,Rural,Mostly Plant-Based,Often,Car,Renewable,House,1445,Rarely,True,3,Low,191,4354,Male,Rarely,Composting,High,5
498,9672120,65,Urban,Balanced,Sometimes,Walk,Non-Renewable,Apartment,2249,Always,False,1,NaN,459,4140,Female,Rarely,Recycling,NaN,2


Study also the variances!

In [20]:
df.var(numeric_only=True)

ParticipantID                     20803.43
Age                                 222.15
HomeSize                         451869.29
SustainableBrands                     0.25
EnvironmentalAwareness                1.80
MonthlyElectricityConsumption     14515.98
MonthlyWaterConsumption         1499667.95
Rating                                2.26
dtype: float64

In [21]:
# fairly similar altogether
synthetic_df.var(numeric_only=True)

ParticipantID                   23861148376637.47
Age                                        227.24
HomeSize                                443394.69
SustainableBrands                            0.25
EnvironmentalAwareness                       1.91
MonthlyElectricityConsumption            15177.49
MonthlyWaterConsumption                1473444.89
Rating                                       2.16
dtype: float64